# MNIST Convolutional Neural Network with TensorFlow GPU

This notebook:

- verifies that TensorFlow detects the NVIDIA GPU;
- loads and preprocesses the MNIST handwritten-digit dataset;
- trains a small convolutional neural network;
- evaluates the model on the test set; and
- plots the training and validation history.

Select the **Python (tf_gpu_wsl)** kernel before running the notebook.

In [ ]:
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from tensorflow.keras import Input, Sequential
from tensorflow.keras.datasets import mnist
from tensorflow.keras.layers import Conv2D, Dense, Flatten, MaxPool2D
from tensorflow.keras.utils import to_categorical

tf.keras.utils.set_random_seed(42)

print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])
print("TensorFlow version:", tf.__version__)

## 1. Verify GPU availability

This cell checks that TensorFlow sees the GPU and reports its model name. Memory growth is enabled before TensorFlow performs any GPU work so the process does not immediately reserve most of the available VRAM.

In [ ]:
gpus = tf.config.list_physical_devices("GPU")

print("Built with CUDA:", tf.test.is_built_with_cuda())
print("Detected GPU count:", len(gpus))
print("Detected GPUs:", gpus)

if not gpus:
    raise RuntimeError(
        "TensorFlow did not detect a GPU. Confirm that the tf_gpu_wsl "
        "kernel is selected and that the CUDA library paths are active."
    )

for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as error:
        print("Could not enable memory growth:", error)

gpu_details = tf.config.experimental.get_device_details(gpus[0])
gpu_name = gpu_details.get("device_name", "Unknown GPU")
compute_capability = gpu_details.get("compute_capability", "Unknown")

print("GPU name:", gpu_name)
print("Compute capability:", compute_capability)

if "RTX 3050" in gpu_name:
    print("GPU model check: PASS — NVIDIA GeForce RTX 3050 detected")
else:
    print("GPU model check: GPU detected, but its name was not RTX 3050")

## 2. Load and preprocess MNIST

MNIST contains 28 × 28 grayscale images of handwritten digits. The images are reshaped to include a single channel dimension and normalized from integer pixel values in `[0, 255]` to floating-point values in `[0, 1]`. The labels are converted to one-hot vectors for categorical cross-entropy.

In [ ]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

x_train = x_train.reshape((-1, 28, 28, 1)).astype("float32") / 255.0
x_test = x_test.reshape((-1, 28, 28, 1)).astype("float32") / 255.0

y_train = to_categorical(y_train, num_classes=10)
y_test = to_categorical(y_test, num_classes=10)

print("Training images:", x_train.shape)
print("Training labels:", y_train.shape)
print("Test images:", x_test.shape)
print("Test labels:", y_test.shape)

## 3. Create training and validation sets

Twenty percent of the original training set is reserved for validation. The test set remains untouched until the final evaluation.

In [ ]:
x_train, x_val, y_train, y_val = train_test_split(
    x_train,
    y_train,
    test_size=0.20,
    random_state=42,
)

print("Final training images:", x_train.shape)
print("Validation images:", x_val.shape)
print("Final training labels:", y_train.shape)
print("Validation labels:", y_val.shape)

## 4. Define the convolutional neural network

The `Flatten` layer is essential. It converts the pooled feature maps from shape `(13, 13, 32)` into one feature vector before the dense classification layers. Without it, the model produces `(13, 13, 10)` predictions instead of one 10-class prediction per image.

In [ ]:
model = Sequential(
    [
        Input(shape=(28, 28, 1)),
        Conv2D(
            filters=32,
            kernel_size=(3, 3),
            activation="relu",
        ),
        MaxPool2D(pool_size=(2, 2)),
        Flatten(),
        Dense(128, activation="relu"),
        Dense(10, activation="softmax"),
    ],
    name="mnist_cnn",
)

model.summary()

## 5. Compile the model

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

## 6. Train the model

The timer measures the complete training call. TensorFlow should log that `/device:GPU:0` corresponds to the NVIDIA GeForce RTX 3050.

In [ ]:
training_start = time.perf_counter()

history = model.fit(
    x_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(x_val, y_val),
    verbose=1,
)

training_seconds = time.perf_counter() - training_start
print(f"Training time: {training_seconds:.2f} seconds")

## 7. Evaluate the model

In [ ]:
test_loss, test_accuracy = model.evaluate(
    x_test,
    y_test,
    verbose=2,
)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4%}")

## 8. Plot training history

The loss curves help show whether the model is still improving, has converged, or is beginning to overfit.

In [ ]:
epochs = range(1, len(history.history["loss"]) + 1)

plt.figure(figsize=(9, 5))
plt.plot(epochs, history.history["loss"], marker="o", label="Training Loss")
plt.plot(epochs, history.history["val_loss"], marker="o", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("MNIST CNN Training and Validation Loss")
plt.xticks(list(epochs))
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(epochs, history.history["accuracy"], marker="o", label="Training Accuracy")
plt.plot(
    epochs,
    history.history["val_accuracy"],
    marker="o",
    label="Validation Accuracy",
)
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("MNIST CNN Training and Validation Accuracy")
plt.xticks(list(epochs))
plt.legend()
plt.tight_layout()
plt.show()

## 9. Confirm that an operation executes on the GPU

This is a small, explicit placement test. The resulting device string should end in `GPU:0`.

In [ ]:
with tf.device("/GPU:0"):
    a = tf.random.normal((2048, 2048))
    b = tf.random.normal((2048, 2048))
    result = tf.matmul(a, b)
    checksum = tf.reduce_sum(result).numpy()

print("GPU name:", gpu_name)
print("Result device:", result.device)
print("Checksum:", checksum)

if "GPU:0" in result.device and "RTX 3050" in gpu_name:
    print("SUCCESS: TensorFlow executed the computation on the RTX 3050.")
else:
    print("The computation or GPU-name verification did not match expectations.")

## 10. Optional: make sample predictions

In [ ]:
sample_count = 10
probabilities = model.predict(x_test[:sample_count], verbose=0)
predicted_labels = np.argmax(probabilities, axis=1)
actual_labels = np.argmax(y_test[:sample_count], axis=1)

print("Predicted:", predicted_labels)
print("Actual:   ", actual_labels)